# 02 — Multi-Pair Feature Engineering

Computes technical indicators (RSI, MACD, ATR, BB, candlestick patterns) per pair.
Adds volatility regime, rolling correlations (with DXY), lag features, and rolling statistics.
Shifts features by 1 day to prevent look-ahead bias.
Saves features.parquet for model training.

**Anti-overfit guard:** Features shifted by 1 day (no future leakage).

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

CURRENCY_PAIRS = [
    "EURUSD", "GBPUSD", "USDJPY", "USDCAD", "AUDUSD",
    "NZDUSD", "USDCHF", "EURGBP", "EURJPY", "EURCHF",
]
PAIR_ID_MAP = {p: i for i, p in enumerate(CURRENCY_PAIRS)}
EXCLUDE_COLS = {"open", "high", "low", "close", "volume", "adj_close", "pair", "pair_id"}

In [ ]:
# Inline indicator functions (no pandas-ta dependency)
def _sma(s, p): return s.rolling(p).mean()
def _ema(s, p): r = s.ewm(span=p, adjust=False).mean(); r.iloc[:p-1] = np.nan; return r
def _rsi(s, p=14):
    d = s.diff(); g = d.where(d>0,0); l = (-d).where(d<0,0)
    ag = g.ewm(span=p, adjust=False).mean(); al = l.ewm(span=p, adjust=False).mean()
    rs = ag / al.replace(0, np.nan); return 100 - (100 / (1 + rs))
def _macd(s, f=12, sl=26, sg=9):
    m = _ema(s,f) - _ema(s,sl); return {"histogram": m - _ema(m,sg)}
def _bb(s, p=20, std=2.0):
    m = _sma(s,p); sd = s.rolling(p).std()
    return {"middle": m, "upper": m+sd*std, "lower": m-sd*std}
def _atr(h, l, c, p=14):
    tr = pd.concat([(h-l).abs(), (h-c.shift(1)).abs(), (l-c.shift(1)).abs()], axis=1).max(axis=1)
    return tr.rolling(p).mean()
def _vol_regime(atr_s, lo=0.3, hi=0.7):
    lt = atr_s.quantile(lo); ht = atr_s.quantile(hi)
    r = pd.Series(1.0, index=atr_s.index); r[atr_s<=lt] = 0.0; r[atr_s>=ht] = 2.0; r[atr_s.isna()] = np.nan
    return r
print("Indicator functions ready")

## Load data from Notebook 01

In [ ]:
df = pd.read_parquet("/kaggle/input/forex-ml-01-data-ingestion/daily.parquet")
print(f"Loaded {len(df)} rows across {df['pair'].nunique()} pairs")

# Also fetch DXY for correlation features
import yfinance as yf
dates = (df.index.min(), df.index.max())
dxy = yf.download("DX-Y.NYB", start=dates[0], end=dates[1], interval="1d", progress=False)
if isinstance(dxy.columns, pd.MultiIndex):
    dxy.columns = dxy.columns.get_level_values(0)
dxy = dxy["Close"].squeeze().rename("DXY")
dxy.index = pd.to_datetime(dxy.index)
print(f"DXY: {len(dxy)} rows")

## Compute features per pair

In [ ]:
def compute_features(sub):
    close, high, low = sub["close"], sub["high"], sub["low"]
    out = sub.copy()

    # Existing technical indicators
    out["RSI"] = _rsi(close, 14)
    macd_res = _macd(close, 12, 26, 9)
    if macd_res is not None:
        out["MACD_Hist"] = macd_res["histogram"]
    bb = _bb(close, 20, 2.0)
    if bb is not None:
        denom = (bb["upper"] - bb["lower"]).clip(lower=1e-8)
        out["BB_PctB"] = (close - bb["lower"]) / denom
        out["BB_Width"] = (bb["upper"] - bb["lower"]) / bb["middle"].clip(lower=1e-8)
    out["ATR"] = _atr(high, low, close, 14)
    out["SMA_20"] = _sma(close, 20)
    out["SMA_50"] = _sma(close, 50)
    ema_fast, ema_slow = _ema(close, 12), _ema(close, 26)
    if ema_slow is not None and (ema_slow != 0).any():
        out["MA_Distance"] = (close - ema_slow) / ema_slow
    out["Body_Ratio"] = (close - sub["open"]) / (high - low).clip(lower=1e-8)
    out["Upper_Wick"] = (high - sub[["open", "close"]].max(axis=1)) / (high - low).clip(lower=1e-8)
    out["Lower_Wick"] = (sub[["open", "close"]].min(axis=1) - low) / (high - low).clip(lower=1e-8)
    out["Log_Returns"] = np.log(close / close.shift(1).clip(lower=1e-8))

    # New: Volatility regime
    out["Vol_Regime"] = _vol_regime(out["ATR"], 0.3, 0.7)

    # New: Rolling skew and kurtosis of returns
    out["Ret_Skew_20"] = out["Log_Returns"].rolling(20).skew()
    out["Ret_Kurt_20"] = out["Log_Returns"].rolling(20).kurt()
    out["Ret_Mean_5"] = out["Log_Returns"].rolling(5).mean()
    out["Ret_Std_5"] = out["Log_Returns"].rolling(5).std()

    # New: Lags of key features
    for lag in [1, 2, 3, 5]:
        out[f"RSI_Lag{lag}"] = out["RSI"].shift(lag)
        out[f"BB_PctB_Lag{lag}"] = out["BB_PctB"].shift(lag)
        out[f"ATR_Lag{lag}"] = out["ATR"].shift(lag)

    return out

groups = []
for pair_name, group in df.groupby("pair", sort=False):
    g = compute_features(group)
    # New: Rolling correlation with DXY
    merged = g.join(dxy, how="left")
    g["Corr_DXY_20"] = merged["Log_Returns"].rolling(20).corr(merged["DXY"].pct_change())
    groups.append(g)

df_feat = pd.concat(groups).sort_index()
print(f"Feature shape: {df_feat.shape}")
feature_cols = [c for c in df_feat.columns if c not in EXCLUDE_COLS]
print(f"Feature count: {len(feature_cols)}")
print(f"Feature columns: {feature_cols}")

## Shift features to prevent look-ahead bias

In [ ]:
df_feat[feature_cols] = df_feat[feature_cols].shift(1)
df_feat = df_feat.dropna()
print(f"After shift+dropna: {len(df_feat)} rows")

In [ ]:
output_path = "/kaggle/working/features.parquet"
df_feat.to_parquet(output_path)
print(f"Saved to {output_path}")
print("Ready for notebook 03 — Model Training")